In [ ]:
import yaml
import pandas as pd
from pathlib import Path
import glob
import numpy as np

# Load the configuration file
CONFIG_FILE_PATH = (
    "config/config.yml"  # Update this path to your actual config location
)

with open(CONFIG_FILE_PATH, "r") as config_file:
    config = yaml.safe_load(config_file)

In [ ]:
def get_preprocess_stats():
    # Get paths from config
    sbs_samples_fp = Path(config["preprocess"]["sbs_samples_fp"])
    phenotype_samples_fp = Path(config["preprocess"]["phenotype_samples_fp"])

    # Load the sample TSV files with pandas
    sbs_samples = pd.read_csv(sbs_samples_fp, sep="\t")
    phenotype_samples = pd.read_csv(phenotype_samples_fp, sep="\t")

    # Count rows in each TSV file
    sbs_input_count = len(sbs_samples)
    phenotype_input_count = len(phenotype_samples)

    # Count output TIFF files in the respective directories
    root_fp = Path(config["all"]["root_fp"])
    sbs_tiff_dir = root_fp / "preprocess" / "images" / "sbs"
    phenotype_tiff_dir = root_fp / "preprocess" / "images" / "phenotype"

    # Count TIFF files in the SBS directory
    sbs_tiff_count = 0
    if sbs_tiff_dir.exists():
        sbs_tiff_count = len(list(sbs_tiff_dir.glob("**/*.tiff")))

    # Count TIFF files in the phenotype directory
    phenotype_tiff_count = 0
    if phenotype_tiff_dir.exists():
        phenotype_tiff_count = len(list(phenotype_tiff_dir.glob("**/*.tiff")))

    return {
        "sbs_tiles": sbs_tiff_count,
        "phenotype_tiles": phenotype_tiff_count,
        "nd2_files": sbs_input_count + phenotype_input_count,
    }

In [3]:
get_preprocess_stats()

{'sbs_tiles': 74646, 'phenotype_tiles': 28890, 'nd2_files': 103536}

In [ ]:
def get_sbs_stats():
    # Extract paths from config
    root_fp = Path(config["all"]["root_fp"])
    sbs_eval_dir = root_fp / "sbs" / "eval"

    # Load segmentation overview data
    seg_overview_files = list(
        sbs_eval_dir.glob("**/segmentation/*__segmentation_overview.tsv")
    )

    total_cells = 0
    if seg_overview_files:
        # Combine all segmentation overview files
        seg_dfs = []
        for file in seg_overview_files:
            df = pd.read_csv(file, sep="\t")
            seg_dfs.append(df)

        if seg_dfs:
            seg_combined = pd.concat(seg_dfs)
            # Sum the final_cells column to get total cells
            total_cells = seg_combined["final_cells"].sum()

    # Load mapping overview data
    mapping_overview_files = list(
        sbs_eval_dir.glob("**/mapping/*__mapping_overview.tsv")
    )

    # Initialize metrics
    one_gene_cells_percent = 0
    one_or_more_barcodes_percent = 0
    total_with_barcode = 0
    total_with_unique_gene = 0

    if mapping_overview_files:
        # Combine all mapping overview files
        map_dfs = []
        for file in mapping_overview_files:
            df = pd.read_csv(file, sep="\t")
            map_dfs.append(df)

        if map_dfs:
            map_combined = pd.concat(map_dfs)

            # Calculate averages across all wells
            one_or_more_barcodes_percent = map_combined[
                "1_or_more_barcodes__percent"
            ].mean()
            one_gene_cells_percent = map_combined["1_gene_cells__percent"].mean()

            # Sum the counts
            total_with_barcode = map_combined["1_or_more_barcodes__count"].sum()
            total_with_unique_gene = map_combined["1_gene_cells__count"].sum()

    return {
        "total_cells": total_cells,
        "percent_cells_with_barcode": one_or_more_barcodes_percent,
        "percent_cells_with_unique_mapping": one_gene_cells_percent,
        "total_with_barcode": total_with_barcode,
        "total_with_unique_gene": total_with_unique_gene,
    }

In [5]:
get_sbs_stats()

{'total_cells': 21368647,
 'percent_cells_with_barcode': 76.49515746773031,
 'percent_cells_with_unique_mapping': 59.46322753831425,
 'total_with_barcode': 16235318,
 'total_with_unique_gene': 12596697}

In [ ]:
def get_phenotype_stats():
    import pyarrow.parquet as pq
    from lib.aggregate.cell_data_utils import DEFAULT_METADATA_COLS

    # Extract paths from config
    root_fp = Path(config["all"]["root_fp"])
    phenotype_eval_dir = root_fp / "phenotype" / "eval" / "segmentation"
    phenotype_parquet_dir = root_fp / "phenotype" / "parquets"

    # Initialize the total cells counter
    total_cells = 0

    # Find all segmentation overview files
    seg_overview_files = list(phenotype_eval_dir.glob("*__segmentation_overview.tsv"))

    if seg_overview_files:
        # Combine all segmentation overview files
        seg_dfs = []
        for file in seg_overview_files:
            df = pd.read_csv(file, sep="\t")
            seg_dfs.append(df)

        if seg_dfs:
            seg_combined = pd.concat(seg_dfs)
            # Sum the final_cells column to get total cells
            total_cells = seg_combined["final_cells"].sum()

    # Get feature count by reading column names from a sample parquet file
    # without loading the entire file into memory
    feature_count = 0
    sample_parquet_files = list(
        phenotype_parquet_dir.glob("**/*__phenotype_cp.parquet")
    )

    if sample_parquet_files:
        # Get just the schema (column names) from the first parquet file
        parquet_file = sample_parquet_files[0]
        parquet_schema = pq.read_schema(parquet_file)
        all_columns = parquet_schema.names

        # Calculate feature count by subtracting metadata columns
        metadata_cols_found = [
            col for col in DEFAULT_METADATA_COLS if col in all_columns
        ]
        feature_count = len(all_columns) - len(metadata_cols_found)

    return {"total_cells": total_cells, "feature_count": feature_count}

In [7]:
get_phenotype_stats()

{'total_cells': 22644651, 'feature_count': 1651}

In [ ]:
def get_merge_stats():
    # Extract paths from config
    root_fp = Path(config["all"]["root_fp"])
    merge_eval_dir = root_fp / "merge" / "eval"

    # Find all cell mapping stats files
    mapping_stats_files = list(merge_eval_dir.glob("**/*__cell_mapping_stats.tsv"))

    if not mapping_stats_files:
        return {"error": "No cell mapping statistics files found"}

    # Initialize counters
    total_mapped_cells = 0
    total_cells = 0
    mapping_percentages = []

    # Process each file
    for file in mapping_stats_files:
        df = pd.read_csv(file, sep="\t")

        # Get counts for mapped and unmapped cells
        mapped_row = df[df["category"] == "mapped_cells"]
        unmapped_row = df[df["category"] == "unmapped_cells"]

        if not mapped_row.empty and not unmapped_row.empty:
            # Add to totals
            mapped_count = mapped_row["count"].iloc[0]
            unmapped_count = unmapped_row["count"].iloc[0]
            file_total = mapped_count + unmapped_count

            total_mapped_cells += mapped_count
            total_cells += file_total

            # Calculate and store mapping percentage for this file
            mapping_percentage = mapped_count / file_total * 100
            mapping_percentages.append(mapping_percentage)

    # Calculate overall percentage and average percentage
    avg_mapping_percentage = np.mean(mapping_percentages) if mapping_percentages else 0

    return {
        "total_cells": total_cells,
        "total_mapped_cells": total_mapped_cells,
        "average_mapping_percentage_across_plates": avg_mapping_percentage,
    }


In [9]:
get_merge_stats()

{'total_cells': 14263564,
 'total_mapped_cells': 12597688,
 'average_mapping_percentage_across_plates': 88.39794927064806}

In our reanalysis, the Aggregate Module effectively reduced technical variation by __% through Typical Variation Normalization, as measured by the mean coefficient of variation of control samples across batches. The final aggregated dataset contained __ distinct perturbations with a median of __ cells per perturbation. 

In [98]:
import pyarrow.dataset as ds
from lib.aggregate.cell_data_utils import (
    DEFAULT_METADATA_COLS,
    split_cell_data,
)
from sklearn.feature_selection import f_classif

def get_aggregate_stats(cell_class, channel_combo):
    # Extract paths from config
    root_fp = Path(config["all"]["root_fp"])
    aggregate_dir = root_fp / "aggregate"

    # Load the aggregated TSV file
    aggregated_path = (
        aggregate_dir / "tsvs" / f"CeCl-{cell_class}_ChCo-{channel_combo}__aggregated.tsv"
    )
    aggregated = pd.read_csv(aggregated_path, sep="\t")

    # Calculate distinct perturbation count and median perturbation cells
    distinct_perturbation_count = len(aggregated["gene_symbol_0"].unique())
    median_perturbation_cells = int(aggregated["cell_count"].median())

    filtered_dir = aggregated_path = aggregate_dir / "parquets"
    filtered_paths = list(
        filtered_dir.glob(f"**/*_CeCl-{cell_class}_ChCo-{channel_combo}__filtered.parquet")
    )

    filtered = ds.dataset(filtered_paths, format="parquet")
    filtered = filtered.to_table(use_threads=True, memory_pool=None).to_pandas()
    filtered = filtered.dropna(axis=1)

    old_control_data = filtered[filtered["gene_symbol_0"].str.startswith("nontargeting")]

    metadata_cols = DEFAULT_METADATA_COLS + [
        "class",
        "confidence",
    ]
    old_control_data["batch"] = old_control_data["plate"].astype(str) + "_" + old_control_data["well"]
    old_control_data = old_control_data.drop(
        columns=metadata_cols,
    )

    # Drop features with zero variance
    old_feature_data = old_control_data.drop(columns=["batch"])
    non_constant_features = old_feature_data.columns[old_feature_data.var() > 0]
    old_feature_data_filtered = old_feature_data[non_constant_features]

    # Now run ANOVA on the filtered features
    X = old_feature_data_filtered.values
    y = old_control_data["batch"].values

    # Run ANOVA F-test
    f_stats, p_vals = f_classif(X, y)

    # Filter out the NaN values
    valid_idx = ~np.isnan(p_vals)
    valid_p_vals = p_vals[valid_idx]
    old_p_vals_median = np.median(valid_p_vals)

    aligned_path = (
        aggregate_dir
        / "parquets"
        / f"CeCl-{cell_class}_ChCo-{channel_combo}__aligned.parquet"
    )
    aligned = pd.read_parquet(aligned_path)
    new_control_data = aligned[aligned["gene_symbol_0"].str.startswith("nontargeting")]
    feature_cols = [
        col for col in new_control_data.columns if col.startswith("PC_")
    ]
    new_control_data["batch"] = new_control_data["plate"].astype(str) + "_" + new_control_data["well"]

    new_control_data = new_control_data[feature_cols + ["batch"]]

    # Separate features and batch labels
    X = new_control_data.drop(columns=["batch"]).values
    y = new_control_data["batch"].values

    # Run ANOVA F-test
    f_stats, p_vals = f_classif(X, y)

    # Filter out the NaN values
    valid_idx = ~np.isnan(p_vals)
    valid_p_vals = p_vals[valid_idx]
    new_p_vals_median = np.median(valid_p_vals)

    return {
        "distinct_perturbation_count": distinct_perturbation_count,
        "median_perturbation_cells": median_perturbation_cells,
        "old_control_p_vals_median": old_p_vals_median,
        "new_control_p_vals_median": new_p_vals_median,
    }

aggregate_stats = get_aggregate_stats("mitotic", "DAPI_COXIV_CENPA_WGA")
aggregate_stats

/tmp/ipykernel_2152597/2587482197.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  old_control_data["batch"] = old_control_data["plate"].astype(str) + "_" + old_control_data["well"]
/tmp/ipykernel_2152597/2587482197.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_control_data["batch"] = new_control_data["plate"].astype(str) + "_" + new_control_data["well"]


{'distinct_perturbation_count': 5296,
 'median_perturbation_cells': 44,
 'old_control_p_vals_median': 3.738557386601617e-09,
 'new_control_p_vals_median': 1.0}